# 12 · Use Case — Revenue & Cashflow Projection

Forecast monthly revenue with confidence bands — the kind of chart founders put
in investor updates. Note: financial series can be **negative** (net cashflow),
so we set `infer_is_positive=False`.

In [ ]:
import numpy as np, torch, timesfm
torch.set_float32_matmul_precision("high")
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(timesfm.ForecastConfig(
    max_context=256, max_horizon=24,
    normalize_inputs=True, use_continuous_quantile_head=True,
    fix_quantile_crossing=True,
    infer_is_positive=False,     # <-- cashflow can go negative
))

In [ ]:
# 3 years of monthly recurring revenue (MRR) with growth + churn wobble
rng = np.random.default_rng(1)
m = np.arange(36)
mrr = (20000 * 1.05**m + rng.normal(0, 1500, m.size)).astype(np.float32)

point, q = model.forecast(horizon=12, inputs=[mrr])
point, q = point[0], q[0]
print("projected MRR (next 12 months):")
for i in range(12):
    print(f"  M+{i+1:2d}: {point[i]:10,.0f}   [80% band {q[i,1]:10,.0f} .. {q[i,9]:10,.0f}]")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
xf = range(len(mrr), len(mrr)+12)
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(range(len(mrr)), mrr, color="tab:blue", label="actual MRR")
ax.plot(xf, point, color="tab:green", lw=2, label="projection")
ax.fill_between(xf, q[:,1], q[:,9], color="tab:green", alpha=0.2, label="80% band")
ax.set_title("MRR projection"); ax.legend()
ax.yaxis.set_major_formatter(lambda x, _: f"${x/1000:.0f}k")
fig.tight_layout(); fig.savefig("mrr_projection.png", dpi=130)
print("saved mrr_projection.png")

### Scenario planning
- **Median** → base case.
- **q10 band** → conservative case for runway calculations.
- **q90 band** → optimistic case for capacity planning.

Feed net cashflow (which can be negative) the same way to project runway.